In [11]:
import time
import json
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Set year to filter
target_year = "2024"

# Setup
driver = webdriver.Chrome()
driver.get("https://media.kkr.com/")
wait = WebDriverWait(driver, 10)

# === STEP 1: Apply year filter ===
# === STEP 1: Apply year filter ===
try:
    print(f"Applying year filter for {target_year}...")
    
    # Wait for the page to load and find the year dropdown
    year_dropdown = wait.until(EC.element_to_be_clickable(
        (By.CSS_SELECTOR, "#mediaNewsLabelGroup .selectric")
    ))
    
    # Click to open the dropdown
    driver.execute_script("arguments[0].click();", year_dropdown)
    time.sleep(4)  # Wait for dropdown to open
    
    # Find and click the target year option
    # The options appear in a dropdown list, usually with class 'selectric-items'
    year_option = wait.until(EC.element_to_be_clickable(
        (By.XPATH, f"//li[contains(text(), '{target_year}')]")
    ))
    driver.execute_script("arguments[0].click();", year_option)
    
    print(f"✅ Successfully selected year {target_year}")
    time.sleep(1)  # Wait for content to reload
    
except (TimeoutException, NoSuchElementException) as e:
    print(f"❌ Error applying year filter: {e}")
    print("Continuing without year filter...")

# === STEP 2: Wait for first press release to load ===
wait.until(EC.presence_of_element_located((By.CLASS_NAME, "press__section")))

# === STEP 3: Load more (limited for testing) ===

while True:
    try:
        load_more = wait.until(EC.presence_of_element_located(
            (By.CSS_SELECTOR, "button.loadmore_button_kkr.active")
        ))
        driver.execute_script("arguments[0].click();", load_more)
        click_count += 1
        print(f"Clicked 'Load more' ({click_count}/{max_clicks})...")
        time.sleep(2)
    except:
        print("No more 'Load more' button or error.")
        break

# === STEP 4: Scrape press releases ===
press_releases = driver.find_elements(By.CLASS_NAME, "press__section")
data = []

for release in press_releases:
    try:
        a_tag = release.find_element(By.CLASS_NAME, "press__link")
        date = a_tag.find_element(By.CLASS_NAME, "press--date").text.strip()
        full_text = a_tag.text.strip()
        title = full_text.replace(date, '').replace('|', '').strip()
        href = a_tag.get_attribute("href")
        full_link = href if href.startswith("http") else f"https://media.kkr.com{href}"

        # Open article page in new tab
        driver.execute_script("window.open(arguments[0]);", full_link)
        time.sleep(1)
        driver.switch_to.window(driver.window_handles[1])

        try:
            wait.until(EC.presence_of_element_located((By.CLASS_NAME, "news_details")))
            paragraphs = driver.find_elements(By.CSS_SELECTOR, ".news_details p")
            article_text = "\n".join(p.text.strip() for p in paragraphs if p.text.strip())
        except Exception as e:
            print("Error getting article text:", e)
            article_text = ""

        driver.close()
        driver.switch_to.window(driver.window_handles[0])

        data.append({
            "date": date,
            "title": title,
            "link": full_link,
            "text": article_text
        })

    except Exception as e:
        print("Error processing a press release:", e)

# === STEP 5: Save to JSON ===
filename = f"kkr_press_releases_{target_year}.json"
with open(filename, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"✅ Saved {len(data)} articles to {filename}")

# === Cleanup ===
driver.quit()


Applying year filter for 2024...
✅ Successfully selected year 2024
Clicked 'Load more' (27/3)...
Clicked 'Load more' (28/3)...
Clicked 'Load more' (29/3)...
Clicked 'Load more' (30/3)...
Clicked 'Load more' (31/3)...
Clicked 'Load more' (32/3)...
Clicked 'Load more' (33/3)...
Clicked 'Load more' (34/3)...
Clicked 'Load more' (35/3)...
No more 'Load more' button or error.
Error processing a press release: list index out of range
Error processing a press release: list index out of range
Error processing a press release: list index out of range
Error getting article text: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Error processing a press release: HTTPConnectionPool(host='localhost', port=50434): Max retries exceeded with url: /session/a4c25fa27a69a5a67acbf1325acae423/window (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x1083c4310>: Failed to establish a new connection: [Errno 61] Connection refused'))
Error 

In [15]:
import time
import json
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

def is_downloadable_link(url):
    """Check if URL points to a downloadable file"""
    downloadable_extensions = ['.pdf', '.doc', '.docx', '.xls', '.xlsx', '.ppt', '.pptx', '.zip', '.rar']
    return any(url.lower().endswith(ext) for ext in downloadable_extensions)

# Set year to filter
target_year = "2024"

# Keep track of download issues
download_issues = []

# Setup
options = webdriver.ChromeOptions()
# Configure Chrome to handle downloads without prompting
prefs = {
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "plugins.always_open_pdf_externally": True  # Don't open PDFs in browser
}
options.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=options)
driver.get("https://media.kkr.com/")
wait = WebDriverWait(driver, 10)

# === STEP 1: Apply year filter ===
try:
    print(f"Applying year filter for {target_year}...")
    
    # Wait for the page to load and find the year dropdown
    year_dropdown = wait.until(EC.element_to_be_clickable(
        (By.CSS_SELECTOR, "#mediaNewsLabelGroup .selectric")
    ))
    
    # Click to open the dropdown
    driver.execute_script("arguments[0].click();", year_dropdown)
    time.sleep(1)  # Wait for dropdown to open
    
    # Find and click the target year option
    # The options appear in a dropdown list, usually with class 'selectric-items'
    year_option = wait.until(EC.element_to_be_clickable(
        (By.XPATH, f"//li[contains(text(), '{target_year}')]")
    ))
    driver.execute_script("arguments[0].click();", year_option)
    
    print(f"✅ Successfully selected year {target_year}")
    time.sleep(2)  # Wait for content to reload
    
except (TimeoutException, NoSuchElementException) as e:
    print(f"❌ Error applying year filter: {e}")
    print("Continuing without year filter...")

# === STEP 2: Wait for first press release to load ===
wait.until(EC.presence_of_element_located((By.CLASS_NAME, "press__section")))

# === STEP 3: Load more (limited for testing) ===

while True:
    try:
        load_more = wait.until(EC.element_to_be_clickable(
            (By.CSS_SELECTOR, "button.loadmore_button_kkr.active")
        ))
        driver.execute_script("arguments[0].click();", load_more)
        click_count += 1
        print(f"Clicked 'Load more' ({click_count}/{max_clicks})...")
        time.sleep(2)
    except (TimeoutException, NoSuchElementException):
        print("No more 'Load more' button or error.")
        break

# === STEP 4: Scrape press releases ===
press_releases = driver.find_elements(By.CLASS_NAME, "press__section")
data = []

print(f"Found {len(press_releases)} press releases to scrape...")

for i, release in enumerate(press_releases, 1):
    try:
        print(f"Processing release {i}/{len(press_releases)}...")
        
        a_tag = release.find_element(By.CLASS_NAME, "press__link")
        date = a_tag.find_element(By.CLASS_NAME, "press--date").text.strip()
        full_text = a_tag.text.strip()
        title = full_text.replace(date, '').replace('|', '').strip()
        href = a_tag.get_attribute("href")
        full_link = href if href.startswith("http") else f"https://media.kkr.com{href}"
        
        # Check if this might be a downloadable file
        if is_downloadable_link(full_link):
            print(f"⚠️ Detected downloadable file for date {date}: {full_link}")
            download_issues.append({
                "date": date,
                "title": title,
                "link": full_link,
                "issue": "Downloadable file detected"
            })
            
            data.append({
                "date": date,
                "title": title,
                "link": full_link,
                "text": "[DOWNLOADABLE FILE - No text content scraped]"
            })
            continue
        
        # Open article page in new tab
        driver.execute_script("window.open(arguments[0]);", full_link)
        
        # Check if new tab was actually opened
        if len(driver.window_handles) > 1:
            driver.switch_to.window(driver.window_handles[1])
            
            # Wait for page to load or detect download
            load_result = wait_for_page_load_or_download(driver, wait_time=8)
            
            if load_result == "page_loaded":
                try:
                    paragraphs = driver.find_elements(By.CSS_SELECTOR, ".news_details p")
                    article_text = "\n".join(p.text.strip() for p in paragraphs if p.text.strip())
                except Exception as e:
                    print(f"Error getting article text for release {i}: {e}")
                    article_text = ""
            
            elif load_result == "download_detected":
                print(f"⚠️ Download detected for date {date}: {full_link}")
                download_issues.append({
                    "date": date,
                    "title": title,
                    "link": full_link,
                    "issue": "Download occurred when accessing link"
                })
                article_text = "[DOWNLOAD DETECTED - No text content scraped]"
            
            else:  # timeout
                print(f"⚠️ Timeout loading page for date {date}: {full_link}")
                download_issues.append({
                    "date": date,
                    "title": title,
                    "link": full_link,
                    "issue": "Page load timeout"
                })
                article_text = "[TIMEOUT - No text content scraped]"
            
            # Close the tab and switch back
            driver.close()
            driver.switch_to.window(driver.window_handles[0])
            
        else:
            # No new tab opened - likely a download started
            print(f"⚠️ No new tab opened for date {date}: {full_link} - likely a download")
            download_issues.append({
                "date": date,
                "title": title,
                "link": full_link,
                "issue": "No new tab opened - download likely occurred"
            })
            article_text = "[NO NEW TAB - Download likely occurred]"
        
        data.append({
            "date": date,
            "title": title,
            "link": full_link,
            "text": article_text
        })
        
    except Exception as e:
        print(f"Error processing press release {i}: {e}")
        continue

# === STEP 5: Save to JSON ===
filename = f"kkr_press_releases_{target_year}.json"
with open(filename, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"✅ Saved {len(data)} articles to {filename}")

# Save download issues log if any occurred
if download_issues:
    issues_filename = f"kkr_download_issues_{target_year}.json"
    with open(issues_filename, "w", encoding="utf-8") as f:
        json.dump(download_issues, f, indent=2, ensure_ascii=False)
    print(f"⚠️ {len(download_issues)} download issues logged to {issues_filename}")
    
    # Print summary of issues
    print("\n📋 Download Issues Summary:")
    for issue in download_issues:
        print(f"  - {issue['date']}: {issue['title'][:50]}... | Issue: {issue['issue']}")

print(f"\n📊 Final Summary:")
print(f"  - Total releases processed: {len(data)}")
print(f"  - Successful text extractions: {len([d for d in data if not d['text'].startswith('[') or 'DOWNLOADABLE FILE' in d['text']])}")
print(f"  - Issues encountered: {len(download_issues)}")

# === Cleanup ===
driver.quit()

Applying year filter for 2024...
✅ Successfully selected year 2024
Clicked 'Load more' (4/3)...
Clicked 'Load more' (5/3)...
Clicked 'Load more' (6/3)...
Clicked 'Load more' (7/3)...
Clicked 'Load more' (8/3)...
Clicked 'Load more' (9/3)...
Clicked 'Load more' (10/3)...
Clicked 'Load more' (11/3)...
Clicked 'Load more' (12/3)...
No more 'Load more' button or error.
Found 145 press releases to scrape...
Processing release 1/145...
Error processing press release 1: name 'wait_for_page_load_or_download' is not defined
Processing release 2/145...
Error processing press release 2: Message: no such element: element not found
  (Session info: chrome=138.0.7204.158); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
0   chromedriver                        0x000000010310f55c cxxbridge1$str$ptr + 2731064
1   chromedriver                        0x0000000103107454 cxxbridge1$str$ptr + 2698032
2 